# Hodgkin–Huxley Modelling: Hands-On Workshop
University of Melbourne Neuroscience PhD Programme, 16 November 2026

This workshop introduces the NEURON simulation environment in Python. Activities 1–3 use a single compartment (a patch of membrane) to explore how ion channels generate action potentials; Activities 4–5 use the multi-compartment retinal ganglion cell model of Sheasby & Fohlmeister (1999) to explore how action potentials start and travel through a real cell shape.

## Activities
1. **Single action potentials:** gating variables, conductances, and the threshold current
2. **Tonic firing:** long current pulses of increasing strength
3. **Altering conductances:** the effect of the Na⁺ conductance
4. **Action potential generation and propagation** in a multi-compartment neuron
5. **Axonal conduction:** conduction velocity and axon diameter

## Setup
Run these two cells first, in order. The first installs NEURON and compiles the ion-channel mechanisms; 
the second imports NEURON, so it must come after.

If anything misbehaves, use **Runtime → Restart session** and run again from the top.

In [ ]:
#@title ▶ Run this first: installs NEURON and fetches the workshop files (~1 min) { display-mode: "form" }
REPO_URL = "https://github.com/jseo977/UniMelb-PhD-Neuroscience-Computational-Modelling-Workshop.git"   # <-- change to your repository

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install NEURON and interactive-plot support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuron==9.0.2", "ipympl"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()

    # Colab loads matplotlib before ipympl is installed, so register the new
    # 'widget' backend with matplotlib (otherwise %matplotlib widget fails)
    try:
        from matplotlib.backends import backend_registry
        new = [(n, m) for n, m in backend_registry._read_entry_points()
               if n.lower() not in backend_registry._backend_to_gui_framework]
        backend_registry._validate_and_store_entry_points(new)
    except Exception:
        pass   # plots will fall back to static images

    # Fetch the workshop files (.mod, morphology, utils) and move into that folder
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)

# Compile the .mod files (skipped if already compiled; delete the x86_64/arm64 folder to force a rebuild)
if not any(os.path.isdir(d) for d in ("x86_64", "arm64", "aarch64")):
    build = subprocess.run(["nrnivmodl"], capture_output=True, text=True)
    if build.returncode != 0:
        print(build.stdout[-3000:], build.stderr[-3000:])
        raise RuntimeError("Compiling the .mod files failed (see messages above)")

print("Setup complete. Working directory:", os.getcwd())


In [ ]:
# Interactive figures (zoom / pan / reset buttons) if available, otherwise static plots
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    get_ipython().run_line_magic('matplotlib', 'inline')
    print("Interactive plots unavailable; using static plots.")
from neuron import h, gui
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

from sheasbyfohlmeister1999_utils import PEAK_HEIGHT, spike_times ,update_rx_3d, plot_final_rgc_with_electrode, run_from_rest

PLOT_ELECTRODE_POSITION = False
TIME_RESOLUTION = 0.05   # ms

plt.rcParams['figure.dpi'] = 100   # widget figures are sized in screen pixels
plt.style.use('seaborn-v0_8-whitegrid')

## The single-compartment model
A single cylindrical compartment with the same size and ion channels as the soma of the retinal ganglion cell used in Activities 4–5 (with a larger leak to stand in for the missing dendrites and axon). Run this cell once before Activities 1–3.

In [ ]:
# A single cylindrical compartment with the soma parameters of ctt3219b.hoc
# (see build_soma_sc in sheasbyfohlmeister1999_utils.py for its size and channel densities).
# If you come back to Activities 1–3 after Activity 4, re-run this cell first.
from sheasbyfohlmeister1999_utils import build_soma_sc, run_soma_sc, find_spikes, spike_half_width

soma_sc = build_soma_sc()
GNA_SC_DEFAULT = soma_sc.gnabar_spike     # S/cm², used by Activity 3
GK_SC_DEFAULT  = soma_sc.gkbar_spike      # S/cm², used by Activity 3


## Activity 1: Single action potentials
### 1a. A brief current pulse
A brief pulse of current (`PULSE_AMP`, `PULSE_DUR` < 5 ms) is injected into the compartment. The figure shows, from top to bottom: the injected current, the membrane potential, the gating variables (m, h, n) and the Na⁺ and K⁺ conductances, all on the same time axis.

- g_Na = ḡ_Na · m³h and g_K = ḡ_K · n⁴ (delayed-rectifier K⁺).

**Questions:** how do the gating variables change over time? Do they change as you expect? Compare the conductance plot with the action potential: when does each conductance rise and fall relative to the upstroke and the downstroke?

In [ ]:
# Activity 1a: a brief current pulse


####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (< 5 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms
TSTOP       = 40     # ms
#############################

r = run_soma_sc(soma_sc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, TSTOP)
g_na = soma_sc.gnabar_spike * r['m']**3 * r['h'] * 1000    # mS/cm²
g_k  = soma_sc.gkbar_spike  * r['n']**4 * 1000             # mS/cm²


# Plot the results
plt.close('all')   # close figures from previous runs
fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True, gridspec_kw={'height_ratios': [1, 2, 2, 2]})
fig.canvas.toolbar_position = 'right'   # keep the zoom/pan toolbar off the axis labels
fig.canvas.header_visible = False
axes[0].plot(r['t'], r['i'], color='k')
axes[0].set_ylabel('IClamp (pA)')
axes[1].plot(r['t'], r['v'], color='k')
axes[1].set_ylabel('Vm (mV)')
axes[2].plot(r['t'], r['m'], label='m (Na act.)', color='#2a78d6')
axes[2].plot(r['t'], r['h'], label='h (Na inact.)', color='#eb6834')
axes[2].plot(r['t'], r['n'], label='n (K act.)', color='#1baf7a')
axes[2].set_ylabel('Gate')
axes[2].legend(loc='upper right')
axes[3].plot(r['t'], g_na, label='gNa', color='#2a78d6')
axes[3].plot(r['t'], g_k,  label='gK (delayed rect.)', color='#1baf7a')
axes[3].set_ylabel('g (mS/cm²)')
axes[3].set_xlabel('Time (ms)')
axes[3].legend(loc='upper right')
plt.tight_layout()
plt.show()

### 1b. Finding the threshold current
For your chosen pulse duration (< 5 ms), the current is increased step by step from a subthreshold starting value until an action potential appears. Set `START_PA` below threshold and `STEP_PA` (no less than 10 pA). Every trace tried is drawn in grey; the first one that fires is drawn in blue.

**Questions:** what is the threshold current for your pulse duration? What happens to the threshold if you make the pulse shorter or longer?

In [ ]:
# Activity 1b: increase the current until the cell fires

####### PARAMETERS ##########
PULSE_DUR = 1        # ms (< 5 ms): your chosen duration
START_PA  = 100      # pA : start below threshold
STEP_PA   = 10       # pA : no less than 10 pA
MAX_PA    = 3000     # pA : give up above this
#############################

plt.close('all')
fig, ax = plt.subplots(figsize=(8, 4))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False

threshold = None
amp = START_PA
while amp <= MAX_PA:
    r = run_soma_sc(soma_sc, amp, PULSE_DUR, delay=10, tstop=40)
    fired = len(find_spikes(r['v'])) > 0
    print(f"{amp:6.0f} pA : {'ACTION POTENTIAL' if fired else 'no spike'}")
    if fired:
        threshold = amp
        ax.plot(r['t'], r['v'], color='#2a78d6', lw=1.8, label=f'{amp:.0f} pA (threshold)')
        break
    ax.plot(r['t'], r['v'], color='grey', lw=0.8, alpha=0.6)
    amp += STEP_PA

ax.plot([], [], color='grey', lw=0.8, label='subthreshold')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Vm (mV)')
ax.set_title(f'Pulse duration {PULSE_DUR} ms')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()
print(f"\nThreshold for a {PULSE_DUR} ms pulse: " +
      (f"{threshold:.0f} pA (to within {STEP_PA} pA)" if threshold else f"no spike up to {MAX_PA} pA"))

## Activity 2: Tonic firing
A long current pulse (`LONG_DUR` > 100 ms) is injected at each strength in `CURRENTS_PA`, from weak to strong. The first figure shows the membrane potential for each current; the second summarises the firing rate, the size of the spikes, and the shape of a single spike from each train (light to dark = weak to strong current).

**Questions:** what do you see as the current increases? What happens to the action potential shape, and why? What happens at the highest currents (*depolarisation block*)? The last column of the table shows the lowest Vm reached near the end of the pulse: during repetitive firing the membrane repolarises between spikes; in depolarisation block it stays depolarised.

In [ ]:
# Activity 2: long current pulses of increasing strength

####### PARAMETERS ##########
LONG_DUR    = 200                                  # ms (> 100 ms)
CURRENTS_PA = [50, 100, 200, 300, 500, 800, 1500]  # pA, weak to strong
DELAY       = 10                                   # ms
TSTOP       = DELAY + LONG_DUR + 40                # ms
#############################

runs_tonic = {}
for amp in CURRENTS_PA:
    runs_tonic[amp] = run_soma_sc(soma_sc, amp, LONG_DUR, DELAY, TSTOP)
shades = plt.cm.Blues(np.linspace(0.35, 0.95, len(CURRENTS_PA)))    # light -> dark = weak -> strong

plt.close('all')
fig, axes = plt.subplots(len(CURRENTS_PA), 1, figsize=(9, 1.3 * len(CURRENTS_PA)), sharex=True, sharey=True)
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
for ax, (amp, r), colour in zip(axes, runs_tonic.items(), shades):
    ax.plot(r['t'], r['v'], color=colour, lw=0.8)
    ax.axvspan(DELAY, DELAY + LONG_DUR, color='orange', alpha=0.08, lw=0)
    ax.set_ylabel(f'{amp} pA', rotation=0, ha='right', va='center')
axes[0].set_title('Vm (mV) for each injected current (shaded: current on)')
axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

rows = []
fig2, ax2 = plt.subplots(1, 3, figsize=(13, 3.6))
fig2.canvas.toolbar_position = 'right'
fig2.canvas.header_visible = False
for (amp, r), colour in zip(runs_tonic.items(), shades):
    pk = find_spikes(r['v'])
    pk = pk[(r['t'][pk] >= DELAY) & (r['t'][pk] <= DELAY + LONG_DUR)]
    rate = len(pk) / (LONG_DUR / 1000)
    peak = r['v'][pk].mean() if len(pk) else np.nan
    width = np.mean([spike_half_width(r['t'], r['v'], i) for i in pk]) if len(pk) else np.nan
    late = (r['t'] >= DELAY + LONG_DUR - 20) & (r['t'] <= DELAY + LONG_DUR)
    v_late = r['v'][late].min()                  # lowest Vm in the last 20 ms of the pulse
    rows.append((amp, len(pk), rate, peak, width, v_late))
    if len(pk):                                   # one spike from the middle of the train
        i_pk = pk[len(pk) // 2]
        sel = (r['t'] >= r['t'][i_pk] - 1.5) & (r['t'] <= r['t'][i_pk] + 2.5)
        ax2[2].plot(r['t'][sel] - r['t'][i_pk], r['v'][sel], color=colour, lw=1.5, label=f'{amp} pA')
amps = [x[0] for x in rows]
ax2[0].plot(amps, [x[2] for x in rows], 'o-', color='k')
ax2[0].set_xlabel('Injected current (pA)')
ax2[0].set_ylabel('Firing rate (Hz)')
ax2[1].plot(amps, [x[3] for x in rows], 'o-', color='k')
ax2[1].set_xlabel('Injected current (pA)')
ax2[1].set_ylabel('Mean spike peak (mV)')
ax2[2].set_xlabel('Time from peak (ms)')
ax2[2].set_ylabel('Vm (mV)')
ax2[2].set_title('One spike from each train', fontsize=10)
ax2[2].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()

print(f"{'current':>9s}{'spikes':>8s}{'rate':>9s}{'mean peak':>11s}{'half-width':>12s}{'lowest Vm,':>13s}")
print(f"{'(pA)':>9s}{'':>8s}{'(Hz)':>9s}{'(mV)':>11s}{'(ms)':>12s}{'last 20 ms':>13s}")
for amp, n, rate, peak, width, v_late in rows:
    print(f"{amp:9.0f}{n:8d}{rate:9.0f}{peak:11.1f}{width:12.2f}{v_late:13.1f}")

## Activity 3: Altering conductances
The maximal Na⁺ conductance (`gnabar_spike`) is scaled by each multiplier in `NA_X_LIST`, and the same short pulse (Activity 1) and long pulse (Activity 2) are injected. Left column: short pulse; right column: long pulse.

**Questions:** what changes as the Na⁺ conductance is reduced to zero, increased moderately, and increased a lot? Why does a very large Na⁺ conductance stop the cell firing during the long pulse? Why does the cell fire *before* the pulse starts (spontaneous action potentials) when the Na⁺ conductance is very large?

`K_X` scales the delayed-rectifier K⁺ conductance (`gkbar_spike`) in the same way, if you want to explore that too.

In [ ]:
# Activity 3: scale the Na conductance and repeat the short and long pulses
NA_X_LIST = [0, 0.5, 1, 2, 6, 10]   # multipliers on the Na conductance
K_X       = 1                       # multiplier on the delayed-rectifier K conductance
SHORT_AMP, SHORT_DUR = 300, 1       # pA, ms (Activity 1)
LONG_AMP,  LONG_DUR  = 300, 200     # pA, ms (Activity 2)
DELAY = 10

plt.close('all')
fig, axes = plt.subplots(len(NA_X_LIST), 2, figsize=(12, 1.4 * len(NA_X_LIST)), sharey=True,
                         gridspec_kw={'width_ratios': [1, 2.2]})
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
print(f"{'NA_X':>6s}{'gNa (S/cm²)':>13s}{'short: spikes':>15s}{'peak (mV)':>11s}{'long: spikes':>14s}{'rate (Hz)':>11s}")
for row, na_x in zip(axes, NA_X_LIST):
    soma_sc.gnabar_spike = GNA_SC_DEFAULT * na_x
    soma_sc.gkbar_spike  = GK_SC_DEFAULT * K_X
    r_s = run_soma_sc(soma_sc, SHORT_AMP, SHORT_DUR, DELAY, DELAY + 30)
    r_l = run_soma_sc(soma_sc, LONG_AMP,  LONG_DUR,  DELAY, DELAY + LONG_DUR + 40)
    row[0].plot(r_s['t'], r_s['v'], color='k', lw=1)
    row[1].plot(r_l['t'], r_l['v'], color='k', lw=0.7)
    row[1].axvspan(DELAY, DELAY + LONG_DUR, color='orange', alpha=0.08, lw=0)
    row[0].set_ylabel(f'×{na_x}', rotation=0, ha='right', va='center')
    ps, pl = find_spikes(r_s['v']), find_spikes(r_l['v'])
    pl = pl[(r_l['t'][pl] >= DELAY) & (r_l['t'][pl] <= DELAY + LONG_DUR)]
    print(f"{na_x:6g}{soma_sc.gnabar_spike:13.3f}{len(ps):15d}"
          f"{(r_s['v'][ps].max() if len(ps) else np.nan):11.1f}{len(pl):14d}{len(pl) / (LONG_DUR / 1000):11.0f}")
soma_sc.gnabar_spike = GNA_SC_DEFAULT        # restore the defaults
soma_sc.gkbar_spike  = GK_SC_DEFAULT
axes[0][0].set_title(f'Short pulse: {SHORT_AMP} pA, {SHORT_DUR} ms')
axes[0][1].set_title(f'Long pulse: {LONG_AMP} pA, {LONG_DUR} ms')
axes[-1][0].set_xlabel('Time (ms)')
axes[-1][1].set_xlabel('Time (ms)')
fig.supylabel('Vm (mV) for each Na multiplier')
plt.tight_layout()
plt.show()

## Activity 4: Action potential generation and propagation
Now the full multi-compartment retinal ganglion cell of Sheasby & Fohlmeister (1999): a soma, a dendritic tree, an axon initial segment (AIS) and a 5.3 mm axon.

### Set up the model
Load the morphology and insert the mechanisms. Run this cell once before Activities 4 and 5.

In [ ]:
# Remove the single-compartment model (Activities 1–3) so it is not part of the full cell
if 'soma_sc' in globals():
    h.delete_section(sec=soma_sc)
    del soma_sc

# Load the model (Colab: from the cloned repo; locally: from neuron_morphology/ in this folder)
import sys
if "google.colab" in sys.modules:
    MORPH_DIR = "/content/UniMelb-PhD-Neuroscience-Computational-Modelling-Workshop/neuron_morphology"
else:
    MORPH_DIR = "neuron_morphology"
h.load_file(f"{MORPH_DIR}/neuron_morphology_ctt3219b.asc")
h.load_file(f"{MORPH_DIR}/neuron_morphology_ctt3219b.hoc")

# Reverse Z coordinates to flip the morphology (soma facing up)
for sec in h.allsec():
    n3d = int(h.n3d(sec=sec))
    if n3d < 2: continue
    xs = [h.x3d(i, sec=sec) for i in range(n3d)]
    ys = [h.y3d(i, sec=sec) for i in range(n3d)]
    zs = [h.z3d(i, sec=sec) for i in range(n3d)]
    ds = [h.diam3d(i, sec=sec) for i in range(n3d)]
    sec.pt3dclear()
    for i in range(n3d):
        sec.pt3dadd(xs[i], ys[i], -zs[i], ds[i])
h.define_shape()

# Centre the cell on the soma
n_soma = int(h.n3d(sec=h.soma))
avg_x = np.mean([h.x3d(i, sec=h.soma) for i in range(n_soma)])
avg_y = np.mean([h.y3d(i, sec=h.soma) for i in range(n_soma)])
avg_z = np.mean([h.z3d(i, sec=h.soma) for i in range(n_soma)])

for sec in h.allsec():
    # Insert extracellular and xtra for field stimulation
    sec.insert('extracellular')
    sec.insert('xtra')
    sec.insert('pas')

    n = int(h.n3d(sec=sec))
    if n < 2:
        continue
    xs = [h.x3d(i, sec=sec) for i in range(n)]
    ys = [h.y3d(i, sec=sec) for i in range(n)]
    zs = [h.z3d(i, sec=sec) for i in range(n)]
    ds = [h.diam3d(i, sec=sec) for i in range(n)]
    sec.pt3dclear()
    for i in range(n):
        sec.pt3dadd(xs[i] - avg_x, ys[i] - avg_y, zs[i] - avg_z, ds[i])

    # Map the NEURON extracellular mechanisms to the xtra pointers
    for seg in sec:
        h.setpointer(seg._ref_e_extracellular, 'ex', seg.xtra)
        h.setpointer(seg._ref_i_membrane, 'im', seg.xtra)

h.ko0_k_ion = 5.0    # mM (extracellular)
h.ki0_k_ion = 145.0  # mM (intracellular)
h.define_shape()

# Electrode position (µm)
el_x, el_y, el_z = 0, 0, 25 + h.soma.diam/2 + 1   # +1 to account for ILM thickness
#el_x, el_y, el_z = 200, 0, 25 + h.soma.diam/2 + 1 # Distal Axon Stimulation
update_rx_3d(el_x, el_y, el_z)

if PLOT_ELECTRODE_POSITION:
    plot_final_rgc_with_electrode(el_x, el_y, el_z)

### Inject a brief pulse into the soma and record throughout the cell
A brief current pulse (`MC_PULSE_AMP`, `MC_PULSE_DUR`) is injected into the soma, and Vm is recorded at the soma, the AIS, three points along a dendrite (blues) and three points along the axon (greens). The darker the colour, the further the site is from the soma.

In [ ]:
# Activity 4: brief current pulse into the soma
MC_PULSE_AMP = 1000   # pA (find the threshold for your pulse duration in the next cell)
MC_PULSE_DUR = 1      # ms
ic_delay     = 5      # ms
ic_amplitude = MC_PULSE_AMP / 1000   # nA (IClamp works in nA)
ic_duration  = MC_PULSE_DUR

h.dt      = TIME_RESOLUTION/10
h.tstop   = 30        # ms (long enough for the spike to reach the far end of the axon)
h.celsius = 33

# Make sure no extracellular stimulus from an earlier run is still playing
from sheasbyfohlmeister1999_utils import stop_extracellular
stop_extracellular()

mc_iclamp = h.IClamp(h.soma(0.5))     # replaces the clamp from any earlier run of this cell
mc_iclamp.delay = ic_delay
mc_iclamp.dur   = ic_duration
mc_iclamp.amp   = ic_amplitude

# Stimulus waveform, for the plot
t_stim = np.arange(0, h.tstop + h.dt, h.dt)
stim_values = np.where((t_stim >= ic_delay) & (t_stim < ic_delay + ic_duration), MC_PULSE_AMP, 0.0)
stim_label  = 'IClamp (pA)'

# Recording vectors
t_vec = h.Vector().record(h._ref_t)
v_rec = {name: h.Vector().record(sec(0.5)._ref_v)
         for name, sec in [('soma', h.soma), ('initseg', h.initseg), ('axon', h.axon)]}
m_soma = h.Vector().record(h.soma(0.5)._ref_m_spike)
h_soma = h.Vector().record(h.soma(0.5)._ref_h_spike)
n_soma = h.Vector().record(h.soma(0.5)._ref_n_spike)

# Vm at the middle of every section, for the interactive dashboard
v_all = {sec.name(): h.Vector().record(sec(0.5)._ref_v) for sec in h.allsec()}

# Dendritic Vm: three points along the longest dendrite, at increasing distance from the soma
from sheasbyfohlmeister1999_utils import dendrite_path_sites
DEND_FRACTIONS = (0.25, 0.55, 0.9)   # fractions of the longest soma-to-tip path (change to move the sites)
dend_sites = [(f'Dendrite {d:.0f} µm', seg, d) for seg, d in dendrite_path_sites(fractions=DEND_FRACTIONS)]
v_dend    = {label: h.Vector().record(seg._ref_v) for label, seg, d in dend_sites}
dend_dist = {label: d for label, seg, d in dend_sites}   # µm from the soma

# Axonal Vm: three points along the axon, at increasing distance from the soma
AXON_FRACTIONS = (0.1, 0.5, 0.9)     # positions along the 5.3 mm axon section
axon_sites = [(f'Axon {h.distance(h.soma(0.5), h.axon(x)) / 1000:.1f} mm', h.axon(x),
               h.distance(h.soma(0.5), h.axon(x))) for x in AXON_FRACTIONS]
v_axon    = {label: h.Vector().record(seg._ref_v) for label, seg, d in axon_sites}
axon_dist = {label: d for label, seg, d in axon_sites}   # µm from the soma

# Colours that encode distance from the soma: dendrites in blues, axon in greens, darker = further
import matplotlib.colors as mcolors
DEND_MAX_UM = 200                                  # longest dendritic path is ~190 µm
AXON_MAX_UM = h.distance(h.soma(0.5), h.axon(1))   # far end of the axon, ~5.5 mm
SITE_COLOURS = {'Soma': '#000000', 'AIS': '#e67e22'}
SITE_COLOURS.update({label: mcolors.to_hex(plt.cm.Blues(0.3 + 0.7 * d / DEND_MAX_UM))
                     for label, seg, d in dend_sites})
SITE_COLOURS.update({label: mcolors.to_hex(plt.cm.Greens(0.3 + 0.7 * d / AXON_MAX_UM))
                     for label, seg, d in axon_sites})

# Recording sites for the dashboard traces: soma, AIS, dendrite and axon sites
dash_sites = ([('Soma', h.soma(0.5)), ('AIS', h.initseg(0.5))]
              + [(label, seg) for label, seg, d in dend_sites]
              + [(label, seg) for label, seg, d in axon_sites])
dash_v = {'Soma': h.Vector().record(h.soma(0.5)._ref_v), 'AIS': h.Vector().record(h.initseg(0.5)._ref_v)}
dash_v.update(v_dend)
dash_v.update(v_axon)

### Find the threshold current
As in Activity 1b: the current is increased step by step until the cell fires. (This uses a coarser time step to run quickly.) Then set `MC_PULSE_AMP` in the cell above to a value at or just above threshold, re-run it, and continue.

**Question:** how does the threshold of the full cell compare with the single compartment in Activity 1b? Why might it be different?

In [ ]:
# Activity 4: threshold current of the full cell for your pulse duration
START_PA = 500      # pA : start below threshold
STEP_PA  = 20       # pA : no less than 10 pA
MAX_PA   = 5000     # pA

dt_saved, tstop_saved, amp_saved = h.dt, h.tstop, mc_iclamp.amp
v_test = h.Vector().record(h.soma(0.5)._ref_v)
t_test = h.Vector().record(h._ref_t)
threshold = None
amp = START_PA
while amp <= MAX_PA:
    mc_iclamp.amp = amp / 1000
    run_from_rest(ic_delay + MC_PULSE_DUR + 15, dt=0.025)
    fired = len(spike_times(np.array(t_test), np.array(v_test))) > 0
    print(f"{amp:6.0f} pA : {'ACTION POTENTIAL' if fired else 'no spike'}")
    if fired:
        threshold = amp
        break
    amp += STEP_PA
h.dt, h.tstop, mc_iclamp.amp = dt_saved, tstop_saved, amp_saved      # restore the settings above
print(f"\nThreshold for a {MC_PULSE_DUR} ms pulse at the soma: " +
      (f"{threshold:.0f} pA (to within {STEP_PA} pA)" if threshold else f"no spike up to {MAX_PA} pA"))

### Run the simulation

In [ ]:
h.finitialize(-62.5)

# Reset calcium to its resting concentration (or else repeated runs will differ)
for sec in h.allsec():
    if h.ismembrane('ca_ion', sec=sec):
        sec.cai = 0.0001
h.fcurrent()
h.frecord_init()

h.continuerun(h.tstop)

t = np.array(t_vec)
v = {name: np.array(vec) for name, vec in v_rec.items()}
vd = {label: np.array(vec) for label, vec in v_dend.items()}   # dendritic Vm
va = {label: np.array(vec) for label, vec in v_axon.items()}   # axonal Vm

peaks, _ = find_peaks(v['soma'], height=PEAK_HEIGHT)
print(f"Somatic spikes: {len(peaks)}  at t = {np.round(t[peaks], 2)} ms")

### Plot the results
From top to bottom: the injected current; Vm at the soma and AIS; Vm along the dendrite; Vm along the axon (soma in grey for reference); and the soma's gating variables and conductances. The table below shows how far each site depolarises, and how long after the soma it peaks.

**Questions:** does the action potential start where you expect? (Zoom in around the first spike to compare the soma and AIS.) How does the spike change as it spreads into the dendrite, and along the axon?

In [ ]:
plt.close('all')   # close figures from previous runs
fig, axes = plt.subplots(6, 1, figsize=(8, 14), sharex=True,
                         gridspec_kw={'height_ratios': [1, 3, 3, 3, 2, 2]})
fig.canvas.toolbar_position = 'right'   # keep the zoom/pan toolbar off the axis labels
fig.canvas.header_visible = False

axes[0].plot(t_stim, stim_values, color='k')
axes[0].set_ylabel(stim_label)

# Soma and AIS
axes[1].plot(t, v['soma'],    color=SITE_COLOURS['Soma'], lw=1.2, label='soma')
axes[1].plot(t, v['initseg'], color=SITE_COLOURS['AIS'],  lw=1.2, label='AIS')
axes[1].set_ylabel('Vm (mV)')
axes[1].legend(loc='upper right')

# Dendrite and axon sites: colour encodes distance from the soma (darker = further); soma in grey
for ax, traces, title in ((axes[2], vd, 'Dendrite Vm (mV)'), (axes[3], va, 'Axon Vm (mV)')):
    ax.plot(t, v['soma'], color='grey', lw=1, alpha=0.5, label='soma')
    for label, trace in traces.items():
        ax.plot(t, trace, color=SITE_COLOURS[label], lw=1.3, label=label)
    ax.set_ylabel(title)
    ax.legend(loc='upper right', fontsize=8)

axes[4].plot(t, np.array(m_soma), label='m (Na act.)')
axes[4].plot(t, np.array(h_soma), label='h (Na inact.)')
axes[4].plot(t, np.array(n_soma), label='n (K act.)')
axes[4].set_ylabel('Gate (soma)')
axes[4].legend(loc='upper right')

# Conductances from the gates (spike.mod): gNa = gnabar*m^3*h, gK = gkbar*n^4 (delayed rectifier)
g_na = h.soma(0.5).gnabar_spike * np.array(m_soma)**3 * np.array(h_soma) * 1000   # mS/cm²
g_k  = h.soma(0.5).gkbar_spike  * np.array(n_soma)**4 * 1000                     # mS/cm²
axes[5].plot(t, g_na, label='gNa')
axes[5].plot(t, g_k,  label='gK (delayed rect.)')
axes[5].set_ylabel('g soma (mS/cm²)')
axes[5].set_xlabel('Time (ms)')
axes[5].legend(loc='upper right')

plt.tight_layout()
plt.show()

# Timing of the first action potential at the AIS vs the soma
ais_pk,  _ = find_peaks(v['initseg'], height=PEAK_HEIGHT)
soma_pk, _ = find_peaks(v['soma'],    height=PEAK_HEIGHT)
if len(ais_pk) and len(soma_pk):
    t_ais, t_soma = t[ais_pk[0]], t[soma_pk[0]]
    print(f"AIS peak:  {t_ais:.3f} ms  ({v['initseg'][ais_pk[0]]:.1f} mV)")
    print(f"Soma peak: {t_soma:.3f} ms  ({v['soma'][soma_pk[0]]:.1f} mV)")
    print(f"Soma - AIS: {(t_soma - t_ais) * 1000:.0f} µs  "
          f"({'AIS first' if t_ais < t_soma else 'soma first' if t_soma < t_ais else 'simultaneous'})")
else:
    print(f"No action potential above {PEAK_HEIGHT} mV at the "
          f"{'AIS' if not len(ais_pk) else 'soma'}, so there is no peak to compare.")

# Spread through the cell: how far each site depolarises during the first somatic spike, and when it peaks
if len(soma_pk):
    t_sp   = t[soma_pk[0]]
    i_rest = int(np.searchsorted(t, ic_delay))                 # just before the current step
    soma_amp = v['soma'][soma_pk[0]] - v['soma'][i_rest]
    print(f"\n{'site':>18s}{'peak':>8s}{'depolarisation':>16s}{'% of soma':>11s}{'peak delay':>12s}")
    print(f"{'':>18s}{'(mV)':>8s}{'(mV)':>16s}{'':>11s}{'(ms)':>12s}")
    print(f"{'soma':>18s}{v['soma'][soma_pk[0]]:8.1f}{soma_amp:16.1f}{100:10.0f}%{0:12.2f}")
    for label, trace in list(vd.items()) + list(va.items()):
        pk_site, _ = find_peaks(trace, height=PEAK_HEIGHT)
        pk_site = pk_site[t[pk_site] >= t_sp - 0.5]             # first spike at or after the somatic one
        if not len(pk_site):
            print(f"{label:>18s}   no spike reached this site before the end of the run")
            continue
        i_pk = pk_site[0]
        amp  = trace[i_pk] - trace[i_rest]
        print(f"{label:>18s}{trace[i_pk]:8.1f}{amp:16.1f}{100 * amp / soma_amp:10.0f}%{t[i_pk] - t_sp:12.2f}")

### Interactive dashboard
The 3D view shows the membrane potential of every section; the traces show the soma, the AIS, the three dendrite sites (◆ on the cell, light to dark blue with distance) and the three axon sites (light to dark green; beyond the edge of the 3D view).

Press **▶ Play**, or drag the time slider. Click and drag the 3D view to rotate it, scroll to zoom, and hover over a ◆ to see its name.

In [ ]:
from sheasbyfohlmeister1999_utils import plot_interactive_dashboard

FRAME_MS = 0.2   # ms between animation frames

dashboard = plot_interactive_dashboard(
    t, v_all,
    sites=[(label, seg, dash_v[label], SITE_COLOURS[label]) for label, seg in dash_sites],
    panels=[('Soma Vm',     ['Soma']),
            ('AIS Vm',      ['AIS']),
            ('Dendrite Vm', list(v_dend)),
            ('Axon Vm',     list(v_axon))],
    gates={'m': m_soma, 'h': h_soma, 'n': n_soma},
    protocol=[('IClamp', ic_delay, ic_delay + ic_duration, 'orange')],
    frame_ms=FRAME_MS,
    title=f"Somatic current injection: {MC_PULSE_AMP} pA for {MC_PULSE_DUR} ms",
)

### How fast does the action potential travel along the axon?
The time at which the spike peaks at each site is plotted against the site's distance from the soma. Along the axon the points fall on a straight line; its slope is 1 / conduction velocity.

**Questions:** which site fires first? How fast does the action potential propagate along the axon (in m/s)? How long would it take to travel the ~4 cm from the eye to the brain at this speed?

In [ ]:
# Activity 4: conduction velocity along the axon
sites = ([('Soma', v['soma'], 0.0), ('AIS', v['initseg'], h.distance(h.soma(0.5), h.initseg(0.5)))]
         + [(label, trace, axon_dist[label]) for label, trace in va.items()])
times = []
for label, trace, d in sites:
    pk, _ = find_peaks(trace, height=PEAK_HEIGHT)
    pk = pk[t[pk] >= ic_delay]
    times.append(t[pk[0]] if len(pk) else np.nan)
times = np.array(times)

plt.close('all')
fig, ax = plt.subplots(figsize=(7, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
for (label, trace, d), tp in zip(sites, times):
    if np.isfinite(tp):
        ax.plot(d / 1000, tp - ic_delay, 'o', ms=9, color=SITE_COLOURS[label], mec='k', mew=0.5, label=label)
axon_pts = [(d, tp) for (label, trace, d), tp in zip(sites, times) if label.startswith('Axon') and np.isfinite(tp)]
cv = None
if len(axon_pts) >= 2:
    d_ax, t_ax = np.array(axon_pts).T
    slope, icpt = np.polyfit(d_ax, t_ax - ic_delay, 1)        # ms per µm
    cv = 1 / slope / 1000                                     # µm/ms -> m/s
    xs = np.array([0, d_ax.max()])
    ax.plot(xs / 1000, slope * xs + icpt, '--', color='grey', lw=1, label=f'fit: {cv:.2f} m/s')
ax.set_xlabel('Distance from soma (mm)')
ax.set_ylabel('Time of spike peak after pulse onset (ms)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for (label, trace, d), tp in zip(sites, times):
    print(f"{label:>14s}: {d:6.0f} µm, " + ("no spike" if np.isnan(tp) else f"peaks {tp - ic_delay:6.2f} ms after the pulse starts"))
first = sites[int(np.nanargmin(times))][0]
print(f"\nFirst site to peak: {first}")
if cv:
    print(f"Conduction velocity along the axon: {cv:.2f} m/s  ->  ~4 cm (eye to brain) would take {40 / cv:.0f} ms")

## Activity 5: Axonal conduction and axon diameter
The axon's diameter is changed to each value in `AXON_DIAMS` (the model's axon is 1 µm), a brief pulse at the soma fires one spike, and its conduction velocity is measured between 0.7 mm and 2.8 mm along the axon. The dashed line shows the cable-theory prediction for an unmyelinated axon: velocity ∝ √diameter.

**Questions:** what happens to the conduction velocity as you thin and thicken the axon? Can you explain this relationship using the capacitance of the larger membrane surface, and the internal (axial) resistance of the thicker axon? Try adding 4 µm: the spike no longer gets into the axon. Why might a thick axon fail to fire when it is driven through the thin (0.8 µm) segment that joins it to the AIS?

In [ ]:
# Activity 5: conduction velocity against axon diameter (run Activity 4's set-up cells first)
AXON_DIAMS = [0.5, 1.0, 1.5, 2.0]     # µm
CV_PULSE_AMP, CV_PULSE_DUR = 2000, 1  # pA, ms : comfortably above threshold
T_ON = 5                              # ms

if 'mc_iclamp' in globals():           # pause Activity 4's clamp
    mc_iclamp.amp = 0
diam0 = h.axon.diam
d_near, d_far = h.distance(h.soma(0.5), h.axon(0.1)), h.distance(h.soma(0.5), h.axon(0.5))
v_near = h.Vector().record(h.axon(0.1)._ref_v)
v_far  = h.Vector().record(h.axon(0.5)._ref_v)
t_cv   = h.Vector().record(h._ref_t)
cv_clamp = h.IClamp(h.soma(0.5))
cv_clamp.delay, cv_clamp.dur, cv_clamp.amp = T_ON, CV_PULSE_DUR, CV_PULSE_AMP / 1000

cvs = []
for diam in AXON_DIAMS:
    h.axon.diam = diam
    run_from_rest(T_ON + 45, dt=0.025)
    tt = np.array(t_cv)
    pn = find_peaks(np.array(v_near), height=PEAK_HEIGHT)[0]
    pf = find_peaks(np.array(v_far),  height=PEAK_HEIGHT)[0]
    cvs.append((d_far - d_near) / (tt[pf[0]] - tt[pn[0]]) / 1000 if len(pn) and len(pf) else np.nan)
h.axon.diam = diam0                    # restore the model
del cv_clamp
if 'mc_iclamp' in globals():
    mc_iclamp.amp = MC_PULSE_AMP / 1000
cvs = np.array(cvs)

plt.close('all')
fig, ax = plt.subplots(figsize=(6.5, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
ax.plot(AXON_DIAMS, cvs, 'o', color='#2ca02c', ms=9, label='model')
if 1.0 in AXON_DIAMS and np.isfinite(cvs[AXON_DIAMS.index(1.0)]):
    dd = np.linspace(min(AXON_DIAMS), max(AXON_DIAMS), 100)
    ax.plot(dd, cvs[AXON_DIAMS.index(1.0)] * np.sqrt(dd), '--', color='grey', label='scales with √diameter')
ax.set_xlabel('Axon diameter (µm)')
ax.set_ylabel('Conduction velocity (m/s)')
ax.legend()
plt.tight_layout()
plt.show()
for d, cv in zip(AXON_DIAMS, cvs):
    print(f"{d:4.1f} µm : " + ("no spike reached 2.8 mm" if np.isnan(cv) else f"{cv:.2f} m/s"))

## Optional extras
Further, self-contained experiments on the multi-compartment cell (run Activity 4's set-up cell first). Run the **set-up cell** first; it pauses the stimulus from Activity 4. Each exercise prints how long it took; most finish in under a minute or two.

In [ ]:
# ── Set-up for the optional extras: run this first (needs the model from Activity 4) ──
# The helper functions are in sheasbyfohlmeister1999_utils.py.
import time
from sheasbyfohlmeister1999_utils import (Recorder, iclamp, pulse_wave, extracellular, stop_extracellular,
                                          reset_stimuli, update_rx_3d, seg_xyz, dendrite_path_sites,
                                          spike_times, find_threshold, scan_threshold, fires_with_pulse,
                                          second_pulse_fires, fires_extracellular, segments, point_above)

EL_HOME = (el_x, el_y, el_z)          # electrode position set up in Activity 4
DT4     = 0.025                       # ms : time step for these exercises

# Vm recorded at four sites; rec4.run(tstop, dt) runs from rest and returns (t, {site: Vm})
rec4 = Recorder({'soma':      h.soma(0.5),
                 'AIS':       h.initseg(0.5),
                 'axon_near': h.axon(0.1),       # 0.7 mm from the soma
                 'axon_far':  h.axon(0.5)})      # 2.8 mm from the soma

reset_stimuli(EL_HOME)                # pauses Activity 4's stimulus (re-run its cell to restore it)
print("Ready.")


### E1. Charging at the injection site: soma vs AIS
The same current step is injected at the soma (recorded at the soma) and then at the AIS (recorded at the AIS), and the two are overlaid from the moment the current switches on. ● marks each spike's threshold (where dV/dt first exceeds 10 V/s on the upstroke).

Injected current first charges the membrane at the injection site: Vm rises at dV/dt = I / C, so a small compartment (small capacitance, C) charges faster than a large one.

**Questions:** which site charges faster at first? Which reaches threshold first, and by how much? Why is the difference in time to threshold much smaller than the difference in capacitance? (Hint: the AIS is connected to the soma.)

In [ ]:
# E1. Charging at the injection site: soma vs AIS
STEP_AMP, STEP_ON, STEP_DUR = 0.3, 5, 15      # nA, ms, ms
DVDT_THRES = 10.0                             # V/s
CASES = [('Inject at soma, record at soma', h.soma(0.5),    'soma', '#000000'),
         ('Inject at AIS, record at AIS',   h.initseg(0.5), 'AIS',  '#e67e22')]

t0_ = time.time()
reset_stimuli(EL_HOME)
plt.close('all')
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
summary, t_last = [], 0
for label, seg, key, colour in CASES:
    clamp = iclamp(seg, STEP_AMP, STEP_ON, STEP_DUR)
    t, v = rec4.run(STEP_ON + STEP_DUR + 5, dt=0.005)      # fine time step to resolve the onset
    del clamp
    trace = v[key]
    dvdt = np.gradient(trace, t)
    i_on = int(np.searchsorted(t, STEP_ON))
    ax.plot(t - STEP_ON, trace, color=colour, lw=1.5, label=label)
    c_pf = seg.area() * seg.cm * 1e-2                                   # µm² × µF/cm² -> pF
    rise = trace[i_on + int(round(0.1 / 0.005))] - trace[i_on - 1]     # mV in the first 0.1 ms
    pk = spike_times(t, trace, STEP_ON)
    if len(pk):
        i_pk = int(np.searchsorted(t, pk[0]))
        i_skip = i_on + int(round(0.2 / 0.005))
        i_th = i_skip + int(np.argmax(dvdt[i_skip:i_pk]))
        while i_th > i_skip and dvdt[i_th - 1] > DVDT_THRES:
            i_th -= 1
        ax.plot(t[i_th] - STEP_ON, trace[i_th], 'o', color=colour, ms=7, mec='k', mew=0.5)
        summary.append((label, c_pf, rise, t[i_th] - STEP_ON, trace[i_th], pk[0] - STEP_ON))
        t_last = max(t_last, pk[0] - STEP_ON)
    else:
        summary.append((label, c_pf, rise, None, None, None))
ax.set_xlim(-0.5, t_last + 1.5)
ax.set_xlabel('Time from current onset (ms)')
ax.set_ylabel('Vm at the injection site (mV)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()
print(f"{'':32s}{'capacitance of':>16s}{'rise in first':>15s}{'time to':>11s}{'threshold':>11s}{'time to':>10s}")
print(f"{'':32s}{'injected segment':>16s}{'0.1 ms':>15s}{'threshold':>11s}{'':>11s}{'peak':>10s}")
print(f"{'':32s}{'(pF)':>16s}{'(mV)':>15s}{'(ms)':>11s}{'(mV)':>11s}{'(ms)':>10s}")
for label, c_pf, rise, t_th, v_th, t_pk in summary:
    if t_th is None:
        print(f"{label:32s}{c_pf:16.3f}{rise:15.2f}   no spike")
    else:
        print(f"{label:32s}{c_pf:16.3f}{rise:15.2f}{t_th:11.2f}{v_th:11.1f}{t_pk:10.2f}")
print(f"({time.time() - t0_:.0f} s)")

### E2. Passive properties: input resistance and membrane time constant
A small hyperpolarising current (too small to open many channels) is injected for 100 ms at the soma, at the AIS and at a dendritic tip (169 µm from the soma), each recorded where the current goes in. The voltage change, ΔV, is measured against a run with no current, so any slow drift at rest cancels out.

- **Input resistance** R_in = ΔV / I (MΩ): how much the voltage changes per unit of current.
- **Time constant** τ: the time to reach 63% of the final ΔV.

**Questions:** why are R_in and τ almost the same at the soma and the AIS, even though the AIS is tiny? Why is the dendritic tip different, especially right after the current switches on?

In [ ]:
# E2. Input resistance and time constant at the soma and the AIS
I_TEST = -0.02     # nA (hyperpolarising)
DELAY, DUR = 20, 100

t0_ = time.time()
reset_stimuli(EL_HOME)
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 4))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
print(f"{'injected at':>14s}{'ΔV (mV)':>10s}{'R_in (MΩ)':>11s}{'τ (ms)':>9s}")
tip_seg = dendrite_path_sites(fractions=(0.9,))[0][0]
v_tip = h.Vector().record(tip_seg._ref_v)
t, v_ctrl = rec4.run(DELAY + DUR + 30)                         # no current: baseline
v_ctrl['tip'] = np.array(v_tip)
for name, seg, key, colour in (('soma', h.soma(0.5), 'soma', 'black'), ('AIS', h.initseg(0.5), 'AIS', '#e67e22'),
                               ('dendrite tip', tip_seg, 'tip', '#08306b')):
    clamp = iclamp(seg, I_TEST, DELAY, DUR)
    t, v = rec4.run(DELAY + DUR + 30)
    v['tip'] = np.array(v_tip)
    del clamp
    dv = v[key] - v_ctrl[key]                              # change caused by the current
    i_on, i_off = np.searchsorted(t, DELAY), np.searchsorted(t, DELAY + DUR)
    dv_ss = dv[i_off - 1]
    i63 = i_on + np.argmax(dv[i_on:i_off] <= 0.632 * dv_ss)
    tau = t[i63] - DELAY
    ax.plot(t - DELAY, dv, color=colour, lw=1.5, label=f'inject and record at {name}')
    print(f"{name:>14s}{dv_ss:10.2f}{dv_ss / I_TEST:11.0f}{tau:9.2f}")
ax.set_xlabel('Time from current onset (ms)')
ax.set_ylabel('ΔV from rest (mV)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"({time.time() - t0_:.0f} s)")

### E3. Strength–duration curve
How much current is needed to fire a spike, for pulses of different length? Short pulses need more current. Two numbers summarise the curve:

- **Rheobase:** the threshold for a very long pulse.
- **Chronaxie:** the pulse length at which the threshold is twice the rheobase. It is a measure of how quickly the cell integrates charge, and is widely used to choose stimulation pulses for neural prostheses.

Set `STIM_SITE` to `'soma'` (intracellular current clamp) or `'electrode'` (cathodic pulse from the extracellular electrode over the soma, as for a retinal implant).

In [ ]:
# E3. Threshold against pulse duration
STIM_SITE = 'soma'                         # 'soma' (current clamp, nA) or 'electrode' (extracellular, µA)
DURATIONS = [0.1, 0.2, 0.5, 1, 2, 5, 10]   # ms
T_ON = 5                                   # ms

t0_ = time.time()
reset_stimuli(EL_HOME)
unit, lo, hi = ('nA', 0.01, 20.0) if STIM_SITE == 'soma' else ('µA', 1.0, 3000.0)
amps = np.geomspace(lo, hi, 12)
# fires_with_pulse(...) -> True if one pulse of amplitude a and duration d fires a spike at the soma
thresholds = np.array([scan_threshold(lambda a: fires_with_pulse(rec4, a, d, STIM_SITE, T_ON), amps)
                       for d in DURATIONS])

rheobase = thresholds[-1]
ok = np.isfinite(thresholds)
chronaxie = np.interp(np.log(2 * rheobase), np.log(thresholds[ok][::-1]), np.log(np.array(DURATIONS)[ok][::-1]))
chronaxie = np.exp(chronaxie)

plt.close('all')
fig, ax = plt.subplots(figsize=(6.5, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
ax.loglog(DURATIONS, thresholds, 'o-', color='k')
ax.axhline(rheobase, color='grey', ls=':', label=f'rheobase ≈ {rheobase:.3g} {unit}')
ax.axhline(2 * rheobase, color='grey', ls='--', label='2 × rheobase')
ax.axvline(chronaxie, color='#d62728', ls='--', label=f'chronaxie ≈ {chronaxie:.2f} ms')
ax.set_xlabel('Pulse duration (ms)')
ax.set_ylabel(f'Threshold ({unit})')
ax.set_title(f"Strength–duration ({'current clamp at soma' if STIM_SITE == 'soma' else 'extracellular electrode'})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
for d, th in zip(DURATIONS, thresholds):
    print(f"{d:6.1f} ms : {th:8.3g} {unit}")
print(f"Rheobase ≈ {rheobase:.3g} {unit}, chronaxie ≈ {chronaxie:.2f} ms   ({time.time() - t0_:.0f} s)")

### E4. Refractory period
Two brief current pulses are delivered to the soma. The first is comfortably above threshold; for each interval between them, the threshold of the **second** pulse is measured.

- **Absolute refractory period:** no second spike is possible, however strong the pulse. Here this is during the first spike itself, so the intervals start just after it (2 ms).
- **Relative refractory period:** a second spike is possible, but needs more current than normal. Watch the threshold fall back towards normal as the interval grows.

**Question:** which ion channel properties (look at the gating variables in Activity 1) set these two periods?

In [ ]:
# E4. Threshold of a second pulse against the interval between pulses
PULSE = 0.5                                        # ms, each pulse
INTERVALS = [2, 2.5, 3, 4, 6, 10, 20]              # ms between pulse onsets
T1 = 5                                             # ms, first pulse

t0_ = time.time()
reset_stimuli(EL_HOME)
# Threshold of a single pulse, then a first pulse at 1.5 × that threshold
th_single = find_threshold(lambda a: fires_with_pulse(rec4, a, PULSE, 'soma', T1, after=20), 0.01, 20.0)
amp1 = 1.5 * th_single
clamp = iclamp(h.soma(0.5), amp1, T1, PULSE)
t_ref, v_ref = rec4.run(T1 + 20)
del clamp
first_spike = spike_times(t_ref, v_ref['soma'], T1)[0]

# Threshold of the second pulse at each interval
# second_pulse_fires(...) -> True if the second pulse (a2 nA, iv ms later) fires a spike of its own
ratios = []
for iv in INTERVALS:
    th2 = find_threshold(lambda a2: second_pulse_fires(rec4, amp1, a2, iv, first_spike, PULSE, T1),
                         0.3 * th_single, 10 * th_single)
    ratios.append(th2 / th_single)
ratios = np.array(ratios)

plt.close('all')
fig, ax = plt.subplots(figsize=(6.5, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
ok = np.isfinite(ratios)
ax.plot(np.array(INTERVALS)[ok], ratios[ok], 'o-', color='k', label='second spike possible')
if (~ok).any():
    ax.plot(np.array(INTERVALS)[~ok], np.full((~ok).sum(), 10), 'x', color='#d62728', ms=10,
            label='no second spike (absolute refractory)')
ax.axhline(1, color='grey', ls=':')
ax.set_xlabel('Interval between pulses (ms)')
ax.set_ylabel('Threshold of 2nd pulse / single-pulse threshold')
ax.set_ylim(0, 10.5)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Single-pulse threshold: {th_single:.2f} nA ({PULSE} ms); first pulse at {amp1:.2f} nA, first spike peaks {first_spike - T1:.1f} ms after it")
for iv, rr in zip(INTERVALS, ratios):
    print(f"{iv:5.1f} ms : " + ("no second spike up to 10× threshold" if not np.isfinite(rr) else f"{rr:.2f} × threshold"))
print(f"({time.time() - t0_:.0f} s)")

### E5. Blocking ion channels
A current step into the soma (0.3 nA for 15 ms), repeated with one type of channel removed, mimicking a drug:

- **TEA-like:** delayed-rectifier K⁺ blocked (`gkbar_spike = 0`).
- **4-AP-like:** A-type K⁺ blocked (`gabar_spike = 0`).
- **Dendritic Na⁺ blocked:** `gnabar_spike = 0` in the dendrites only.

**Questions:** which block broadens the spike, which changes the delay to the first spike, and which stops the cell firing? What does the last one tell you about the role of the dendrites in this model?

In [ ]:
# E5. Channel block
STEP_AMP, STEP_ON, STEP_DUR = 0.3, 5, 15    # nA, ms, ms
CONDITIONS = {
    'Control':                     [],
    'TEA-like (delayed-rect. K)':  [('all',  'gkbar_spike')],
    '4-AP-like (A-type K)':        [('all',  'gabar_spike')],
    'Dendritic Na blocked':        [('dend', 'gnabar_spike')],
}
COLOURS = ['black', '#1f77b4', '#2ca02c', '#d62728']

t0_ = time.time()
reset_stimuli(EL_HOME)
plt.close('all')
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
print(f"{'condition':>28s}{'spikes':>8s}{'1st spike (ms)':>16s}{'peak (mV)':>11s}{'half-width (ms)':>17s}")
for (name, blocks), colour in zip(CONDITIONS.items(), COLOURS):
    saved = []
    for where, param in blocks:                         # block
        for seg in segments(where):        # every segment, or only the dendrites
            saved.append((seg, param, getattr(seg, param)))
            setattr(seg, param, 0.0)
    clamp = iclamp(h.soma(0.5), STEP_AMP, STEP_ON, STEP_DUR)
    t, v = rec4.run(STEP_ON + STEP_DUR + 10)
    del clamp
    for seg, param, val in saved:                       # restore
        setattr(seg, param, val)
    vs = v['soma']
    ax.plot(t, vs, color=colour, lw=1.3, label=name)
    st = spike_times(t, vs, STEP_ON)
    if len(st):
        i_pk = np.searchsorted(t, st[0])
        base = vs[np.searchsorted(t, st[0] - 1.0)]
        half = base + (vs[i_pk] - base) / 2
        above = vs >= half
        lo, hi = i_pk, i_pk
        while lo > 0 and above[lo - 1]:
            lo -= 1
        while hi < len(vs) - 1 and above[hi + 1]:
            hi += 1
        print(f"{name:>28s}{len(st):8d}{st[0] - STEP_ON:16.2f}{vs[i_pk]:11.1f}{(hi - lo) * DT4:17.2f}")
    else:
        print(f"{name:>28s}{0:8d}{'no spike':>16s}")
ax.axvspan(STEP_ON, STEP_ON + STEP_DUR, color='orange', alpha=0.1)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Soma Vm (mV)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"({time.time() - t0_:.0f} s)")

### E6. Extracellular stimulation: electrode position and pulse shape
The extracellular electrode (as in a retinal implant) is moved over different parts of the cell, and the threshold current is measured for three pulse shapes: **cathodic** (negative), **anodal** (positive) and **biphasic** (cathodic then anodal, charge-balanced as used clinically). A response counts only if a spike travels 2.8 mm down the axon, i.e. the signal that would reach the brain.

A strong cathodic pulse depolarises the membrane right under the electrode but hyperpolarises the axon on either side of it, which can stop the spike from escaping (*cathodic block*). The table marks pulse shapes that work at threshold but fail again at the highest current (\*).

**Questions:** where is the cell easiest to activate? Which pulse shape has the lowest threshold, and is that what you expected? What is the cost of using charge-balanced biphasic pulses? Why might very strong stimulation fail to send a signal to the brain?

In [ ]:
# E6. Extracellular threshold against electrode position and pulse shape
PHASE_DUR = 0.5                          # ms per phase
SHAPES    = ['cathodic', 'anodal', 'biphasic']
T_ON      = 5
HEIGHT    = EL_HOME[2]                   # electrode height above the cell (µm), as in Section 3

POSITIONS = {'over soma':            EL_HOME,
             'over AIS':             point_above(h.initseg(0.5), HEIGHT),
             'over axon (0.25 mm)':  point_above(h.axon(0.03), HEIGHT),
             'over axon (1 mm)':     point_above(h.axon(0.16), HEIGHT)}

# fires_extracellular(...) -> True if a spike reaches 2.8 mm down the axon ('axon_far')
fires = lambda a, shape: fires_extracellular(rec4, a, shape, PHASE_DUR, T_ON, key='axon_far')

AMPS = np.geomspace(5, 3000, 14)     # µA, stepped through from the bottom

t0_ = time.time()
reset_stimuli(EL_HOME)
table, blocked = {}, {}
for pos_name, pos in POSITIONS.items():
    update_rx_3d(*pos)
    table[pos_name]   = [scan_threshold(lambda a: fires(a, s), AMPS) for s in SHAPES]
    blocked[pos_name] = [np.isfinite(th) and not fires(AMPS[-1], s) for th, s in zip(table[pos_name], SHAPES)]
update_rx_3d(*EL_HOME)               # restore the electrode

plt.close('all')
fig, ax = plt.subplots(figsize=(8, 4.5))
fig.canvas.toolbar_position = 'right'
fig.canvas.header_visible = False
x = np.arange(len(POSITIONS))
w = 0.26
for k, (shape, colour) in enumerate(zip(SHAPES, ['#1f77b4', '#d62728', '#7f7f7f'])):
    vals = [table[p][k] for p in POSITIONS]
    ax.bar(x + (k - 1) * w, np.nan_to_num(vals, nan=0), w, color=colour, label=shape)
ax.set_xticks(x)
ax.set_xticklabels(list(POSITIONS), fontsize=9)
ax.set_ylabel('Threshold (µA)')
ax.set_title(f'Extracellular threshold, {PHASE_DUR} ms per phase (no bar = no spike up to 3000 µA)', fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()
print(f"{'electrode':>22s}" + ''.join(f"{s:>12s}" for s in SHAPES) + "   threshold (µA); * = blocked again at 3000 µA")
for p, vals in table.items():
    print(f"{p:>22s}" + ''.join(f"{'none':>12s}" if np.isnan(val) else f"{val:11.0f}{'*' if b else ' '}"
                                 for val, b in zip(vals, blocked[p])))
print(f"({time.time() - t0_:.0f} s)")